# Module A3: Face Recognition and Visit Logging

This notebook demonstrates face recognition using face embedding extraction (via a pre-trained ResNet18 network) on cropped face regions. It covers registering a new customer, querying face embeddings using cosine similarity, and logging customer visits with timestamps.

In [ ]:
import cv2
import numpy as np
import os
import pickle
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from datetime import datetime
import matplotlib.pyplot as plt

import sys
sys.path.append('../')
from app.services import cv_utils

print('PyTorch device:', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

## 1. Define Face Embedding Extractor
We use a pre-trained ResNet18 model as a feature extractor. We extract the output of the average pooling layer (512 dimensions) and L2-normalize the vector.

In [ ]:
class FaceEmbedder:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.features.to(self.device)
        self.features.eval()
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def get_embedding(self, face_bgr):
        # Convert OpenCV BGR to RGB
        rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(rgb)
        tensor = self.transform(pil_img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            emb = self.features(tensor).squeeze().cpu().numpy()
        # L2 Normalize
        norm = np.linalg.norm(emb)
        if norm > 0:
            emb = emb / norm
        return emb

embedder = FaceEmbedder()
print('Face embedder initialized.')

## 2. Register Consenting Test Faces
Let's crop the face from `data/lena.jpg` and register it as 'Lena' in our database.

In [ ]:
image_path = '../data/lena.jpg'
img = cv_utils.read_image(image_path)
faces = cv_utils.detect_faces(img)

face_db = {}
if len(faces) > 0:
    x, y, w, h = faces[0]
    lena_face = img[y:y+h, x:x+w]
    embedding = embedder.get_embedding(lena_face)
    face_db['Lena'] = [embedding]
    print("Successfully registered 'Lena' face with embedding length:", len(embedding))
else:
    print('No face detected to register!')

## 3. Verify Cosine Similarity for Matching
We perturb the registered face (using slight resizing/blurring) to simulate a query frame and verify if cosine similarity correctly identifies the match.

In [ ]:
# Simulate a query image of Lena with slight Gaussian blur
query_face = cv_utils.blur_image(lena_face, kernel_size=5, method='gaussian')
query_emb = embedder.get_embedding(query_face)

def recognize_face(query_emb, face_db, threshold=0.75):
    best_name = 'Unknown'
    best_sim = 0.0
    for name, embeddings in face_db.items():
        for db_emb in embeddings:
            # Cosine similarity since embeddings are normalized is just the dot product
            sim = np.dot(query_emb, db_emb)
            if sim > best_sim:
                best_sim = sim
                if sim >= threshold:
                    best_name = name
    return best_name, best_sim

match_name, match_sim = recognize_face(query_emb, face_db)
print(f'Query Result: recognized as {match_name} with similarity {match_sim:.4f}')

## 4. Log Visit
Log recognized customer visits in a CSV file.

In [ ]:
def log_visit(name, log_path='../data/customer_visits.csv'):
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    file_exists = os.path.exists(log_path)
    with open(log_path, 'a') as f:
        if not file_exists:
            f.write('name,timestamp\n')
        f.write(f'{name},{timestamp}\n')
    print(f'Logged visit for {name} at {timestamp}')

log_visit(match_name)

## 5. Save Face Database

In [ ]:
os.makedirs('../app/models', exist_ok=True)
db_path = '../app/models/face_db.pkl'
with open(db_path, 'wb') as f:
    pickle.dump(face_db, f)
print(f'Face database successfully saved to: {db_path}')